# 评论批量推理测试

- 使用 `examples/comment.md` 作为模型 instruction。
- 从 `/home/dream/ProjectDATA/Testset/csvs/3_1_Panshan_Scenic_Area_Jizhou_District_Tianjin.csv` 逐条读取评论。
- 将 JSON 结果按列展开后保存到 `examples/output/comment_instruction_results.csv`。

In [4]:
from __future__ import annotations

import json
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI

PROJECT_ROOT = Path("/home/dream/Study/26p1/DSproject/ProjectNew")
PROMPT_PATH = PROJECT_ROOT / "examples" / "comment.md"
INPUT_CSV = Path("/home/dream/ProjectDATA/Testset/csvs/3_1_Panshan_Scenic_Area_Jizhou_District_Tianjin.csv")
OUTPUT_CSV = PROJECT_ROOT / "examples" / "output" / "comment_instruction_results3-5.csv"

BASE_URL = "http://localhost:8000/v1"
API_KEY = "EMPTY"
MODEL_NAME = "Qwen/Qwen3.5-4B"
MAX_ROWS = 100
TEMPERATURE = 0.7
MAX_TOKENS = 4096
REQUEST_INTERVAL_SECONDS = 0.0
KEEP_SOURCE_COMMENT_COLUMN = False


def load_instruction(path: Path) -> str:
    return path.read_text(encoding="utf-8").strip()


def find_column(df: pd.DataFrame, candidates: list[str]) -> str:
    normalized = {str(col).strip().lower(): col for col in df.columns}
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
        lowered = candidate.strip().lower()
        if lowered in normalized:
            return normalized[lowered]
    raise KeyError(f"未找到列：{candidates}。实际列为：{list(df.columns)}")


def message_content_to_text(content) -> str:
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts: list[str] = []
        for item in content:
            if isinstance(item, dict):
                text = item.get("text")
                if text:
                    parts.append(str(text))
            elif item is not None:
                parts.append(str(item))
        return "\n".join(parts)
    return str(content)


def extract_json_text(raw_text: str) -> str:
    cleaned = raw_text.strip()
    if not cleaned:
        raise ValueError("模型返回为空")

    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        if lines and lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].startswith("```"):
            lines = lines[:-1]
        cleaned = "\n".join(lines).strip()
        if cleaned.lower().startswith("json"):
            cleaned = cleaned[4:].strip()

    try:
        json.loads(cleaned)
        return cleaned
    except json.JSONDecodeError:
        pass

    object_start = cleaned.find("{")
    array_start = cleaned.find("[")
    starts = [index for index in (object_start, array_start) if index != -1]
    if not starts:
        raise ValueError("返回内容中未找到 JSON 起始符")
    start = min(starts)
    end = max(cleaned.rfind("}"), cleaned.rfind("]"))
    if end == -1 or end <= start:
        raise ValueError("返回内容中未找到 JSON 结束符")
    candidate = cleaned[start:end + 1]
    json.loads(candidate)
    return candidate


def parse_json_response(raw_text: str) -> dict:
    parsed = json.loads(extract_json_text(raw_text))
    if not isinstance(parsed, dict):
        raise ValueError("模型返回不是 JSON 对象")
    return parsed


def is_empty_list_like(value) -> bool:
    if value is None or value == 0:
        return True
    if isinstance(value, str):
        return value.strip() in {"", "0", "[]", "[0]", "null", "None"}
    if isinstance(value, list):
        if not value:
            return True
        return all(is_empty_list_like(item) for item in value)
    return False


def normalize_text_list(value) -> list[str]:
    if is_empty_list_like(value):
        return []
    if isinstance(value, list):
        items: list[str] = []
        for item in value:
            if is_empty_list_like(item):
                continue
            if isinstance(item, (dict, list)):
                items.append(json.dumps(item, ensure_ascii=False))
            else:
                text = str(item).strip()
                if text:
                    items.append(text)
        return items
    text = str(value).strip()
    return [text] if text else []


def format_string_list(value) -> str:
    items = normalize_text_list(value)
    return "[]" if not items else "; ".join(items)


def format_scalar_text(value) -> str:
    if value is None:
        return ""
    return str(value).strip()


def flatten_comment_result(parsed: dict) -> dict:
    text_analysis = parsed.get("text_analysis") or {}
    sentiment = text_analysis.get("comment_sentiment") or {}
    return {
        "emotions": format_string_list(text_analysis.get("emotions")),
        "influence_of_emotions": format_scalar_text(text_analysis.get("influence_of_emotions")),
        "text_species_mentions": format_string_list(text_analysis.get("text_species_mentions")),
        "feeling_correlated_to_text_species": format_string_list(text_analysis.get("feeling_correlated_to_text_species")),
        "text_activities_or_facilities": format_string_list(text_analysis.get("text_activities_or_facilities")),
        "feeling_correlated_to_text_activities_or_facilities": format_string_list(
            text_analysis.get("feeling_correlated_to_text_activities_or_facilities")
        ),
        "comment_sentiment_score_0_to_1": sentiment.get("score_0_to_1", ""),
    }


instruction = load_instruction(PROMPT_PATH)
source_df = pd.read_csv(INPUT_CSV)
username_col = find_column(source_df, ["用户名", "username", "user_name"])
comment_col = find_column(source_df, ["评论", "comment", "comments"])

if MAX_ROWS is not None:
    source_df = source_df.head(MAX_ROWS).copy()

print(f"待处理评论数：{len(source_df)}")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
records: list[dict] = []

for row_number, (_, row) in enumerate(source_df.iterrows(), start=1):
    username = "" if pd.isna(row[username_col]) else str(row[username_col]).strip()
    comment_text = "" if pd.isna(row[comment_col]) else str(row[comment_col]).strip()
    raw_response = ""
    base_record = {
        "source_row": row_number,
        "用户名": username,
    }
    if KEEP_SOURCE_COMMENT_COLUMN:
        base_record["评论"] = comment_text

    if not comment_text:
        records.append(
            {
                **base_record,
                **flatten_comment_result({}),
                "raw_response": raw_response,
                "parse_ok": False,
                "error": "空评论，已跳过",
            }
        )
        continue

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": instruction},
                {
                    "role": "user",
                    "content": (
                        "Analyze the following comment.\n"
                        "All output text must be in English only.\n"
                        "Never copy Chinese text from the source comment.\n"
                        "For any list with no result, return [] exactly.\n\n"
                        f"Comment:\n{comment_text}"
                    ) 
                },
            ],
            max_tokens=MAX_TOKENS,
            temperature=TEMPERATURE,
            top_p=0.8,
            presence_penalty=1.5,
            extra_body={
                "top_k": 20,
                "chat_template_kwargs": {"enable_thinking": False},}, 
        )
        
        msg = response.choices[0].message
        content = msg.content
  
        raw_response = message_content_to_text(content)
        

        parsed = parse_json_response(raw_response)
        
        records.append(
            {
                **base_record,
                **flatten_comment_result(parsed),
                "raw_response": raw_response,
                "parse_ok": True,
                "error": "",
            }
        )
    except Exception as exc:
        records.append(
            {
                **base_record,
                **flatten_comment_result({}),
                "raw_response": raw_response,
                "parse_ok": False,
                "error": str(exc),
            }
        )

    if REQUEST_INTERVAL_SECONDS:
        time.sleep(REQUEST_INTERVAL_SECONDS)

    if row_number % 10 == 0 or row_number == len(source_df):
        print(f"已完成 {row_number}/{len(source_df)} 条评论")

results_df = pd.DataFrame(records)
ordered_columns = [
    "source_row",
    "用户名",
    *( ["评论"] if KEEP_SOURCE_COMMENT_COLUMN else [] ),
    "emotions",
    "influence_of_emotions",
    "text_species_mentions",
    "feeling_correlated_to_text_species",
    "text_activities_or_facilities",
    "feeling_correlated_to_text_activities_or_facilities",
    "comment_sentiment_score_0_to_1",
    "raw_response",
    "parse_ok",
    "error",
]
results_df = results_df.reindex(columns=ordered_columns)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"结果已保存到：{OUTPUT_CSV}")
results_df.head()

待处理评论数：100
已完成 10/100 条评论
已完成 20/100 条评论
已完成 30/100 条评论
已完成 40/100 条评论
已完成 50/100 条评论
已完成 60/100 条评论
已完成 70/100 条评论
已完成 80/100 条评论
已完成 90/100 条评论
已完成 100/100 条评论
结果已保存到：/home/dream/Study/26p1/DSproject/ProjectNew/examples/output/comment_instruction_results3-5.csv


,source_row,用户名,emotions,influence_of_emotions,text_species_mentions,feeling_correlated_to_text_species,text_activities_or_facilities,feeling_correlated_to_text_activities_or_facilities,comment_sentiment_score_0_to_1,raw_response,parse_ok,error
0,1,1天津市蓟州区盘山_Winnie0222,Excitement; Contentment; Amusement,Discounted tickets,[],[],climbing; viewing scenery; checking in spots; ...,climbing - relaxing; viewing scenery - broaden...,0.85,"{\n ""text_analysis"": {\n ""emotions"": [\n ...",True,
1,2,1天津市蓟州区盘山_不可触碰撞的鱼,Awe; Wonder; Appreciation,Scenic beauty,[],[],Hiking; Climbing steps; Wu Yun Tower; Small th...,Wu Yun Tower - Awe; Small theater ruins - Hist...,0.95,"{\n ""text_analysis"": {\n ""emotions"": [\n ...",True,
2,3,1天津市蓟州区盘山_不可触碰撞的鱼,Appreciation; Contentment; Awe,Artistic craftsmanship,[],[],sculptures; relief carvings; mountain paths; f...,sculptures - admiration; relief carvings - viv...,0.92,"{\n ""text_analysis"": {\n ""emotions"": [\n ...",True,
3,4,1天津市蓟州区盘山_不可触碰撞的鱼,Admiration; Awe; Contentment,Historical grandeur,Ginkgo biloba,Ginkgo biloba - Ancient majesty,Temple visit; Tourism; Religious pilgrimage; C...,Temple visit - Spiritual reverence; Tourism - ...,0.95,"{\n ""text_analysis"": {\n ""emotions"": [\n ...",True,
4,5,1天津市蓟州区盘山_不可触碰撞的鱼,Admiration; Contentment; Awe,Historical grandeur,[],[],visiting temple; cable car ride; stone inscrip...,visiting temple - Awe; cable car ride - Scenic...,0.85,"{\n ""text_analysis"": {\n ""emotions"": [\n ...",True,
